In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import pynapple as nap
from sympy import false
from tqdm import tqdm
import random
import matplotlib
from pathlib import Path

# matplotlib.use('TkAgg')  # Use Agg backend f|or interactive plotting
# matplotlib.use('module://matplotlib_inline.backend_inline')

from Utils.json_tools import read_formatted_json
from Utils.kilosort_utils import get_neuro_summary
from Utils.load_files import get_interval_pairs
from Utils.plotting import get_tuning_curve_for_cluster, plot_hd_tuning_curve, plot_tuning_curves_for_cluster, apply_light_plot_style

file_names = read_formatted_json("./file_names.json")
session_info_filename: str = file_names["session_info_filename"]
sync_data_filename: str = file_names["sync_data_filename"]
head_direction_filename: str = file_names["head_direction_filename"]
valid_events_filename: str = file_names["valid_events_filename"]
interval_table_filename: str = file_names["interval_table_filename"]
kilosort_info_filename: str = file_names["kilosort_info_filename"]

In [ ]:
base_dir = r"/mnt/senzailab/Kai/#Recording/m15"

# base_dir = '/Volumes/SenzaiLab/Kai/#Recording/m12'
date: str | int = "260630"
multi_recording: bool = True

num_of_rec: int = 1

headplate_name: str = 'hp4'

is_photodiode = lambda num_of_rec, multi_recording: (
    True
    if num_of_rec == 1 and multi_recording
    else False if num_of_rec == 2 and multi_recording
    else None)

# is_photodiode = False

recording_freq: int = 120
phase_key = "baseline"

num_shuffle = 500
is_overwrite_shuffle_data: bool = False

num_of_bins_in_hd = 24


with tqdm(total=5, desc="Reading files", unit="%",
          bar_format="{l_bar}{bar}| {n}/{total} [{percentage:3.0f}%]") as pbar:
    subfolder_filler = f"{date}_{num_of_rec}"

    base_dir = f"{base_dir}/{date}/{subfolder_filler}" if multi_recording else f"{base_dir}/{date}"

    kilosort_dir = Path(base_dir) / f"kilosort"

    data_dir: str = f"{base_dir}/data"

    session_info: dict = read_formatted_json(f"{data_dir}/{session_info_filename}.json")["session_info"]
    probe_count = session_info["probe_count"]
    pbar.update(1)

    sync_data: dict = read_formatted_json(f"{data_dir}/{sync_data_filename}.json")
    pbar.update(1)

    interval_table = pd.read_csv(f"{base_dir}/data/{interval_table_filename}.csv")
    pbar.update(1)

    ADC_sample_rate: float = session_info["ADC_sample_rate"]

    interval_pairs_all = np.asarray(
        get_interval_pairs(interval_table,
                           phase_key=phase_key), dtype=float)
    interval_pairs = interval_pairs_all
    recording_start_time = interval_pairs_all[0][0]



    hd_content = read_formatted_json(f"{data_dir}/processed/{head_direction_filename}.json")
    pbar.update(1)

    hd_raw = hd_content[headplate_name].get('head_direction_deg')
    hd = np.asarray(hd_raw, dtype=float)
    # hd = hd - 24.610919452941175
    hd = (hd % 360).tolist()

    hd_frames = hd_content[headplate_name].get('frames')
    hd_s = np.array(hd_frames) / recording_freq

    hd_s_adjusted = hd_s + recording_start_time

    HD_tsd = nap.Tsd(t=hd_s_adjusted, d=hd)
    pbar.update(1)


interval_pairs = interval_pairs_all[[0]]
interval_pairs

In [ ]:
hd_summary_A = get_neuro_summary(
    base_dir=base_dir,
    kilosort_dir=kilosort_dir,
    probe_name="A",
    session_info=session_info,
    interval_pairs=interval_pairs,
    HD_tsd=HD_tsd,
    kilosort_info_filename=kilosort_info_filename,
    num_of_bins_in_hd=num_of_bins_in_hd,
    num_shuffle=num_shuffle,
    is_overwrite_shuffle_data=is_overwrite_shuffle_data,
    is_return_tsgroup=True,
    is_return_time_support=True,
    is_return_hd_tcs=True,
    is_return_cluster_KSLabel=True,
    is_return_hd_cells=True,
    is_return_classical_hd_cells=True,
)


tsgroup_A = hd_summary_A["tsgroup"]
time_support_A = hd_summary_A["time_support"]
hd_tuning_curves_A = hd_summary_A["hd_tuning_curves"]
cluster_KSLabel_sort_A = hd_summary_A["cluster_KSLabel"]
hd_cells_A = hd_summary_A["hd_cells"]
classical_hd_cells_A = hd_summary_A["classical_hd_cells"]



In [ ]:
sub_min = random.randint(int((time_support_A / 1)[0][0]), int((time_support_A / 1)[0][1]))

# epoch = nap.IntervalSet(interval_all)
epoch = nap.IntervalSet(np.asarray([[2250, 2600]]))

decoded, proba_feature = nap.decode_bayes(
    tuning_curves=hd_tuning_curves_A,
    data=tsgroup_A_all,
    epochs=epoch,
    sliding_window_size=4,
    bin_size=0.02,
)

plt.plot(HD_tsd.restrict(epoch), label="True")
plt.plot(decoded, label="Decoded")
plt.legend()

In [ ]:

plt.figure(figsize=(60, 60))

# Convert xarray tuning curves to a bins x units DataFrame for plotting
angle_dim = next(dim for dim in hd_tuning_curves_A.dims if dim != "unit")
tc = hd_tuning_curves_A.transpose(angle_dim, "unit").to_pandas().astype(float)

# Peak-normalize each cell (per column) to [0, 1]
denom = tc.max(axis=0).replace(0, np.nan)  # avoid divide-by-zero
tc_norm = tc.divide(denom, axis=1).fillna(0.0)

xtick_goal = 6
steps = len(tc_norm) / xtick_goal
# arrangement = (np.arange(0, len(tc_norm), step = steps)).astype(int)
arrangement = (np.arange(0, len(tc_norm), step=steps)).astype(int)

multiplier = 360 / num_of_bins_in_hd

# plt.imshow(hd_tuning_curves.T, aspect='auto',
#            interpolation='nearest',
#            cmap='jet')

plt.imshow(tc_norm.T, aspect='auto', interpolation='nearest', cmap='jet')

plt.xticks(arrangement, (arrangement * multiplier))
plt.colorbar()
plt.show()

In [ ]:
# Sort neurons by preferred direction, occupancy-aware

fig, ax = plt.subplots(figsize=(24, 18))

angle_dim = next(dim for dim in hd_tuning_curves_A.dims if dim != "unit")
tc = hd_tuning_curves_A.transpose(angle_dim, "unit").to_pandas().astype(float)

# HD occupancy in the same epoch used for the tuning curves
hd_ep = HD_tsd.restrict(time_support_A)
occupancy, _ = np.histogram(hd_ep.values, bins=num_of_bins_in_hd, range=(0, 360))
valid_bins = occupancy > 0

# Keep unsampled bins as NaN so they show up as blank/gray
tc_occ = tc.copy()
tc_occ.loc[~valid_bins, :] = np.nan

# Peak-normalize each neuron using only sampled bins
peak = tc_occ.max(axis=0, skipna=True).replace(0, np.nan)
tc_norm = tc_occ.divide(peak, axis=1)

# Preferred direction from sampled bins only
pref_dir = tc_norm.idxmax(axis=0, skipna=True)

# Put neurons with no valid peak at the end
valid_cells = pref_dir.notna()
sorted_cols = (
        pref_dir[valid_cells]
        .sort_values(kind="stable")
        .index
        .tolist()
        + pref_dir[~valid_cells].index.tolist()
)

tc_norm_sorted = tc_norm.loc[:, sorted_cols]

cmap = plt.cm.jet.copy()
cmap.set_bad(color="lightgray")

im = ax.imshow(
    tc_norm_sorted.T,
    aspect="auto",
    # interpolation="gaussian",
    interpolation="nearest",
    cmap=cmap,
    origin="lower",
    extent=[0, 360, 0, tc_norm_sorted.shape[1]],
)

ax.set_xlabel("Head direction (degrees)")
ax.set_ylabel("Neurons sorted by preferred direction")
ax.set_title("HD tuning curves")

ax.set_xticks(np.linspace(0, 360, 7))
ax.set_yticks([])

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Peak-normalized firing rate")

plt.tight_layout()
plt.show()


In [ ]:
plt.figure()
for cluster_id in cluster_KSLabel_sort_A[0][1]:
    curve = get_tuning_curve_for_cluster(hd_tuning_curves_A, cluster_id)
    plt.plot(curve.index.values, curve.values)
plt.xlabel("Orientation (degrees)")
plt.ylabel("Firing rate (Hz)")
plt.xlim(0, 360)
plt.show()

# plt.figure()
# for cluster_id in cluster_KSLabel_sort_B[0][1]:
#     curve = get_tuning_curve_for_cluster(hd_tuning_curves_B, cluster_id)
#     plt.plot(curve.index.values, curve.values)
# plt.xlabel("Orientation (degrees)")
# plt.ylabel("Firing rate (Hz)")
# plt.xlim(0, 360)
# plt.show()


In [ ]:
list = [
    108, 114, 124, 127, 131, 133, 134, 137, 139, 141, 153, 156, 160, 267, 270, 274, 275, 279, 280, 285, 287, 291, 294, 296, 307, 318, 423, 459,
]



In [ ]:
apply_light_plot_style()

for cluster_id in hd_summary_A["cluster_KSLabel"][0][1]:
    plots = plot_hd_tuning_curve(
        hd_tuning_curves_A,
        cluster_id,
        is_return_line_plot=False,
        is_return_polar_plot=True,
        ring_levels=(1,),
        is_save=True,
        clockwise=False,
        is_smooth=True,
        smooth_sigma=1,
        save_path=f"{data_dir}/tc_curve/ProbeA"
    )


for cluster_id in hd_summary_B["cluster_KSLabel"][0][1]:
    plots = plot_hd_tuning_curve(
        hd_tuning_curves_B,
        cluster_id,
        is_return_line_plot=False,
        is_return_polar_plot=True,
        ring_levels=(1,),
        is_save=True,
        clockwise=False,
        is_smooth=True,
        smooth_sigma=1,
        save_path=f"{data_dir}/tc_curve/ProbeB"
    )



In [ ]:
apply_light_plot_style()

for cluster_id in list:
    plots = plot_hd_tuning_curve(
        hd_tuning_curves_A,
        cluster_id,
        is_return_line_plot=False,
        is_return_polar_plot=True,
        ring_levels=(1,),
        is_save=False,
        clockwise=False,
        is_smooth=True,
        smooth_sigma=1
    )

In [ ]:
from scipy.ndimage import gaussian_filter1d


In [ ]:
target_tc = hd_tuning_curves_A


selectedSpikeCounts = target_tc.transpose("unit", angle_dim:='0').values[np.isin(target_tc.coords["unit"].values.tolist(), list), :]
selectedUnitIDs = np.array(list)

unitsSpikeCounts_smooth = gaussian_filter1d(
    selectedSpikeCounts,
    sigma=1.5,
    axis=1
)

rank_idx = np.argsort(maxIndex:= np.argmax(unitsSpikeCounts_smooth, axis=1))
unitsSpikeCounts_smooth_sorted = unitsSpikeCounts_smooth[rank_idx, :]
targetList_sorted = selectedUnitIDs[rank_idx]


plot_tuning_curves_for_cluster(
    unitsSpikeCounts_smooth_sorted,
    targetList_sorted,
    isNormalize=True,
    isHeatmap=True,
    xinDeg=True,

)

plot_tuning_curves_for_cluster(
    unitsSpikeCounts_smooth_sorted,
    targetList_sorted,
    isNormalize=True,
    isLineplot=True,
    offset=0.5,
    xinDeg=True,
    plotSize=(10, 15)
)

In [ ]:
for cluster_id in hd_cells_B.cluster_id:
    plots = plot_hd_tuning_curve(
        hd_tuning_curves_B,
        cluster_id,
        is_return_line_plot=False,
        is_return_polar_plot=True,
        ring_levels=(1,),
        is_save=False
    )

In [ ]:
for cluster_id in classical_hd_cells_B.cluster_id:
    plots = plot_hd_tuning_curve(
        hd_tuning_curves_B,
        cluster_id,
        is_return_line_plot=False,
        is_return_polar_plot=True,
        ring_levels=(1,),
    )

In [ ]:
hd_summary_A_all = get_neuro_summary(
    base_dir=base_dir,
    kilosort_dir=kilosort_dir,
    probe_name="A",
    session_info=session_info,
    interval_pairs=interval_pairs,
    HD_tsd=HD_tsd,
    kilosort_info_filename=kilosort_info_filename,
    num_of_bins_in_hd=num_of_bins_in_hd,
    num_shuffle=num_shuffle,
    is_overwrite_shuffle_data=is_overwrite_shuffle_data,
    is_return_tsgroup=True,
    is_return_time_support=True,
    is_return_hd_tcs=True,
    is_return_cluster_KSLabel=True,
    is_return_classical_hd_cells=True,
    save_path=f"{base_dir}/data/tc_summary_ProbeA_ori.csv",
)

hd_tuning_curves_A_all = hd_summary_A_all["hd_tuning_curves"]
tsgroup_A_all = hd_summary_A_all["tsgroup"]

classical_hd_cells_A_all = hd_summary_A_all["classical_hd_cells"]


In [ ]:
# Raster + real HD, in degrees

ep = nap.IntervalSet(
    start=100,
    end=150,
)  # Change this interval if needed

spikes = tsgroup_B
spikes_hd = spikes[hd_cells_B["cluster_id"].astype(int).tolist()]
tuning_curves = hd_tuning_curves_B.sel(unit=spikes_hd.keys())

angle_dim = next(dim for dim in tuning_curves.dims if dim != "unit")
pref_ang = tuning_curves.idxmax(dim=angle_dim)

plt.subplots(figsize=(12, 6))
plt.rc("font", size=12)

for n in spikes_hd.keys():
    preferred_hd = pref_ang.sel(unit=n).item()
    color = plt.cm.hsv(preferred_hd / 360)

    plt.plot(
        spikes[n].restrict(ep).fillna(preferred_hd),
        "|",
        color=color,
    )

plt.plot(
    HD_tsd.restrict(ep),
    "-",
    color="black",
    linewidth=2,
    label="real HD",
)

plt.legend(loc="upper left")
plt.xlabel("Time (s)")
plt.ylabel("Head direction (deg)")
plt.show()

In [ ]:

def iter_sliding_windows(time_support, window_size, step_size):
    for start, end in np.asarray(time_support.values, dtype=float):
        last_start = end - window_size
        if last_start < start:
            continue

        current = start
        while current <= last_start + 1e-9:
            yield float(current), float(current + window_size)
            current += step_size


def preferred_direction_from_curve(curve, method="peak"):
    curve = curve.dropna()
    if curve.empty:
        return np.nan

    if method == "peak":
        return float(curve.idxmax())

    if method == "circular_mean":
        angles_rad = np.deg2rad(curve.index.to_numpy(dtype=float))
        weights = curve.to_numpy(dtype=float)

        x = np.sum(weights * np.cos(angles_rad))
        y = np.sum(weights * np.sin(angles_rad))

        if np.isclose(x, 0.0) and np.isclose(y, 0.0):
            return np.nan

        return float(np.mod(np.rad2deg(np.arctan2(y, x)), 360.0))

    raise ValueError("method must be 'peak' or 'circular_mean'")


def compute_preferred_direction_over_time(
        tsgroup,
        hd_tsd,
        cluster_ids: list[int],
        time_support,
        window_size: float,
        step_size: float | None = None,
        nb_bins: int = 360,
        minmax: tuple[float, float] = (0.0, 360.0),
        min_spikes: int = 20,
        min_occupied_bins: int = 20,
        method: str = "peak",
        return_tuning: bool = False,
):
    if step_size is None:
        step_size = window_size

    cluster_ids = [int(cid) for cid in cluster_ids]
    if len(cluster_ids) == 0:
        raise ValueError("cluster_ids is empty")

    available_ids = {int(cid) for cid in np.asarray(tsgroup.index)}
    missing_ids = sorted(set(cluster_ids) - available_ids)
    if missing_ids:
        raise KeyError(f"cluster ids not found in tsgroup: {missing_ids}")

    preferred_rows = []
    spike_rows = []
    info_rows = []
    tuning_by_window = {}

    for start, end in iter_sliding_windows(time_support, window_size, step_size):
        ep = nap.IntervalSet(start=start, end=end)
        window_center = (start + end) / 2.0

        window_group = tsgroup[cluster_ids].restrict(ep)
        window_hd = hd_tsd.restrict(ep)

        occupancy, _ = np.histogram(window_hd.values, bins=nb_bins, range=minmax)
        occupied_bins = occupancy > 0
        occupied_bin_count = int(np.count_nonzero(occupied_bins))

        tuning_curves = None
        if occupied_bin_count >= min_occupied_bins:
            tuning_curves = nap.compute_1d_tuning_curves(
                window_group,
                hd_tsd,
                nb_bins=nb_bins,
                ep=ep,
                minmax=minmax,
            ).astype(float)

            # mark unsampled angle bins as NaN
            tuning_curves.loc[~occupied_bins, :] = np.nan

            if return_tuning:
                tuning_by_window[window_center] = tuning_curves.copy()

        preferred_row = {}
        spike_row = {}

        for cid in cluster_ids:
            spike_count = len(window_group[cid])
            spike_row[cid] = int(spike_count)

            if tuning_curves is None or spike_count < min_spikes:
                preferred_row[cid] = np.nan
            else:
                preferred_row[cid] = preferred_direction_from_curve(
                    tuning_curves[cid],
                    method=method,
                )

        preferred_rows.append(pd.Series(preferred_row, name=window_center))
        spike_rows.append(pd.Series(spike_row, name=window_center))
        info_rows.append(
            {
                "window_center": window_center,
                "window_start": start,
                "window_end": end,
                "occupied_bins": occupied_bin_count,
                "hd_samples": int(len(window_hd)),
            }
        )

    preferred_df = pd.DataFrame(preferred_rows).sort_index()
    spike_count_df = pd.DataFrame(spike_rows).sort_index()
    window_info_df = pd.DataFrame(info_rows).set_index("window_center").sort_index()

    preferred_df.index.name = "window_center"
    spike_count_df.index.name = "window_center"

    if return_tuning:
        return preferred_df, spike_count_df, window_info_df, tuning_by_window

    return preferred_df, spike_count_df, window_info_df


In [ ]:
def plot_preferred_direction_over_time(
        preferred_df,
        cluster_ids: list[int] | None = None,
        *,
        unwrap: bool = False,
        figsize=(14, 5),
        ax=None,
        is_show_legend: bool = False
):
    if cluster_ids is None:
        cluster_ids = [int(cid) for cid in preferred_df.columns]
    else:
        cluster_ids = [int(cid) for cid in cluster_ids]

    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.figure

    x = preferred_df.index.to_numpy(dtype=float)

    for cid in cluster_ids:
        y = preferred_df[cid].to_numpy(dtype=float)

        if unwrap:
            y_plot = np.full_like(y, np.nan)
            valid = np.isfinite(y)
            if np.any(valid):
                y_plot[valid] = np.rad2deg(np.unwrap(np.deg2rad(y[valid])))
        else:
            y_plot = y

        ax.plot(x, y_plot, marker="o", linewidth=1.5, markersize=4, label=f"cluster {cid}")

    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Preferred direction (deg)")
    ax.set_title("Preferred direction over time")
    ax.grid(True, alpha=0.3)

    if not unwrap:
        ax.set_ylim(0, 360)
        ax.set_yticks(np.arange(0, 361, 60))

    if is_show_legend:
        ax.legend()
    return fig, ax


def plot_preferred_direction_debug(
        cluster_id: int,
        preferred_df,
        spike_count_df,
        window_info_df,
        unwrap: bool = False,
        figsize=(14, 9),
):
    fig, axes = plt.subplots(3, 1, figsize=figsize, sharex=True)

    x = preferred_df.index.to_numpy(dtype=float)
    y = preferred_df[cluster_id].to_numpy(dtype=float)

    if unwrap:
        y_plot = np.full_like(y, np.nan)
        valid = np.isfinite(y)
        if np.any(valid):
            y_plot[valid] = np.rad2deg(np.unwrap(np.deg2rad(y[valid])))
    else:
        y_plot = y

    axes[0].plot(x, y_plot, marker="o")
    axes[0].set_ylabel("Pref dir (deg)")
    axes[0].set_title(f"Cluster {cluster_id}")

    if not unwrap:
        axes[0].set_ylim(0, 360)
        axes[0].set_yticks(np.arange(0, 361, 60))

    axes[0].grid(True, alpha=0.3)

    axes[1].plot(x, spike_count_df[cluster_id].to_numpy(dtype=float), marker="o", color="C1")
    axes[1].set_ylabel("Spike count")
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(
        window_info_df.index.to_numpy(dtype=float),
        window_info_df["occupied_bins"].to_numpy(dtype=float),
        marker="o",
        color="C2",
    )
    axes[2].set_xlabel("Time (s)")
    axes[2].set_ylabel("Occupied bins")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    return fig, axes


In [ ]:
cluster_ids = [109]
window_size = 60  # seconds
step_size = 30  # seconds

preferred_df, spike_count_df, window_info_df, tuning_by_window = compute_preferred_direction_over_time(
    tsgroup=tsgroup_A,
    hd_tsd=HD_tsd,
    cluster_ids=cluster_ids,
    time_support=time_support_A,
    window_size=window_size,
    step_size=step_size,
    nb_bins=360,
    min_spikes=20,
    min_occupied_bins=20,
    method="peak",  # or "circular_mean"
    return_tuning=True,
)

preferred_df.head()


In [ ]:
plot_preferred_direction_over_time(
    preferred_df,
    cluster_ids=cluster_ids,
    unwrap=False,  # True can help if the line jumps around 0/360
)
plt.show()
